In [6]:
import os
import osmnx as ox
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point, LineString
import matplotlib.pyplot as plt
import folium
import ast
from shapely import wkt
from shapely.geometry import shape
import json

In [2]:

G = ox.graph_from_place("San Francisco County, California, USA", network_type="all")
nodes, edges = ox.graph_to_gdfs(G)
edges = edges.to_crs("EPSG:32610")
nodes = nodes.to_crs("EPSG:32610")
print(f"Nodes: {len(nodes)}, Edges: {len(edges)}")

KeyboardInterrupt: 

In [ ]:
# --- Plot the street network ---
fig, ax = ox.plot_graph(G, figsize=(12, 12), node_size=0,
                         edge_linewidth=0.5, show=True)

In [ ]:
# --- Filter edges to only those with a populated footway tag ---
footway_edges = edges[edges["footway"].notna()] if "footway" in edges.columns else edges.iloc[0:0]
print(f"Footway edges: {len(footway_edges)}")

# --- Plot only footway-tagged segments ---
fig, ax = plt.subplots(figsize=(12, 12))
footway_edges.plot(ax=ax, linewidth=0.8, color="steelblue")
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
# Inspect an edge's attributes
u, v, key, data = list(G.edges(data=True, keys=True))[0]
print(data)

In [ ]:


# ── CONFIG ─────────────────────────────────────────────────────────────────────
PARQUET_PATH = "Output/San_Francisco_County_California_USA_network.parquet"  # adjust as needed

CENTER_LAT = 37 + 46/60 + 16.6/3600   # 37°46'16.6"N
CENTER_LON = -(122 + 25/60 + 27.1/3600)  # 122°25'27.1"W
ZOOM = 19

# ── LOAD ───────────────────────────────────────────────────────────────────────
df = pd.read_parquet(PARQUET_PATH)
print(f"Loaded {len(df)} rows")

# ── GEOMETRY HELPER ────────────────────────────────────────────────────────────
def parse_geom(val):
    """Parse a geometry from WKT string, GeoJSON dict/string, or shapely object."""
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return None
    try:
        if hasattr(val, 'geom_type'):
            return val
        if isinstance(val, dict):
            return shape(val)
        s = str(val).strip()
        if not s or s.lower() in ('none', 'nan', 'null'):
            return None
        if s.startswith('{'):
            return shape(json.loads(s))
        return wkt.loads(s)
    except Exception:
        return None

def geom_to_latlons(geom):
    """Return list of [lat, lon] pairs for LineString / MultiLineString."""
    if geom is None:
        return []
    if geom.geom_type == 'LineString':
        return [[c[1], c[0]] for c in geom.coords]
    if geom.geom_type == 'MultiLineString':
        coords = []
        for line in geom.geoms:
            coords.extend([[c[1], c[0]] for c in line.coords])
        return coords
    return []

# ── BUILD MAP ──────────────────────────────────────────────────────────────────
m = folium.Map(location=[CENTER_LAT, CENTER_LON], zoom_start=ZOOM,
                tiles='CartoDB positron')

counts = {'street': 0, 'sidewalk': 0, 'bikeway': 0}

for _, row in df.iterrows():

    # ── Streets (red) ────────────────────────────────────────────────────────
    street_geom = parse_geom(row.get('street_geometry'))
    coords = geom_to_latlons(street_geom)
    if coords:
        name = row.get('name', '') or ''
        hw   = row.get('highway', '') or ''
        folium.PolyLine(
            coords, color='red', weight=3, opacity=0.85,
            tooltip=f"Street: {name} ({hw})"
        ).add_to(m)
        counts['street'] += 1

    # ── Sidewalks (blue) ─────────────────────────────────────────────────────
    for side in ('left', 'right'):
        sw_geom = parse_geom(row.get(f'sidewalk_{side}_geometry'))
        coords = geom_to_latlons(sw_geom)
        if coords:
            presence = row.get(f'sidewalk_{side}_presence', '')
            width    = row.get(f'sidewalk_{side}_width', '')
            folium.PolyLine(
                coords, color='blue', weight=2, opacity=0.75,
                tooltip=f"Sidewalk {side}: presence={presence}, width={width}"
            ).add_to(m)
            counts['sidewalk'] += 1

    # ── Bikeways (green) ─────────────────────────────────────────────────────
    for side in ('left', 'right'):
        for num in (1, 2):
            bk_geom = parse_geom(row.get(f'bikeway_{side}_{num}_geometry'))
            coords = geom_to_latlons(bk_geom)
            if coords:
                bk_type = row.get(f'bikeway_{side}_{num}_type', '')
                folium.PolyLine(
                    coords, color='green', weight=2, opacity=0.75,
                    tooltip=f"Bikeway {side}-{num}: {bk_type}"
                ).add_to(m)
                counts['bikeway'] += 1

# ── CENTER MARKER ──────────────────────────────────────────────────────────────
folium.Marker(
    [CENTER_LAT, CENTER_LON],
    popup='Market Street Block',
    icon=folium.Icon(color='orange', icon='map-marker')
).add_to(m)

# ── LEGEND ─────────────────────────────────────────────────────────────────────
legend_html = f"""
<div style="position:fixed;bottom:30px;left:30px;z-index:9999;
            background:white;padding:10px 14px;border:2px solid #aaa;
            border-radius:6px;font-size:13px;line-height:1.8;">
<b>Legend</b><br>
<span style="color:red">&#9644;</span> Streets ({counts['street']})<br>
<span style="color:blue">&#9644;</span> Sidewalks ({counts['sidewalk']})<br>
<span style="color:green">&#9644;</span> Bikeways ({counts['bikeway']})
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

print(f"Streets: {counts['street']}  |  Sidewalks: {counts['sidewalk']}  |  Bikeways: {counts['bikeway']}")

# ── FORCE HTML RENDER (bypasses VS Code iframe block) ─────────────────────────
html_str = m.get_root().render()

# Write to a temp file and open in browser, AND inline-render in the notebook
tmp_path = os.path.join(tempfile.gettempdir(), "parkximity_map.html")
with open(tmp_path, "w", encoding="utf-8") as f:
    f.write(html_str)
print(f"Map saved to: {tmp_path}")

# Inline display: wraps the full HTML in a srcdoc iframe that VS Code allows
display(HTML(f"""
<iframe srcdoc="{html_str.replace('"', '&quot;')}"
        width="100%" height="600px"
</iframe>
"""))

#   Notes:
#   - Adjust PARQUET_PATH to point to your .parquet file (relative to the notebook or absolute).
#   - Geometry columns parsed are street_geometry, sidewalk_left/right_geometry, and bikeway_left/right_1/2_geometry — matching the schema
#    at lines 392-396.
#   - parse_geom handles WKT strings, GeoJSON dicts/strings, and native shapely objects, so it'll work regardless of how geometries were  
#   serialized.
#   - The map renders inline in Jupyter via the final m.

Loaded 184236 rows
Streets: 0  |  Sidewalks: 0  |  Bikeways: 0


NameError: name 'tempfile' is not defined